# Frogeindrip: Exploratory Data Analysis & Evidence Reconciliation
**Objective:** Connect to DuckDB Lakehouse, validate verified Square baseline metrics, analyze customer RFM distributions, and examine cohort retention dynamics.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from frogeindrip.config import DB_PATH, VERIFIED_EVIDENCE

print("Libraries loaded successfully.")

## 1. Verified Evidence Constants
Reviewing the ground-truth Square screenshots:

In [ ]:
for group, data in VERIFIED_EVIDENCE.items():
    print(f"=== {group.upper()} ===")
    for k, v in data.items():
        print(f"  {k}: {v}")

## 2. Connect to DuckDB Lakehouse & Query Monthly Mart
Querying `mart_monthly_performance` for MoM growth and gross margins.

In [ ]:
conn = duckdb.connect(str(DB_PATH), read_only=True)
monthly_df = conn.execute("SELECT * FROM mart_monthly_performance ORDER BY year_month").fetchdf()
display(monthly_df)
conn.close()

## 3. Customer RFM Segmentation Distribution
Evaluating spend concentration across customer loyalty tiers.

In [ ]:
conn = duckdb.connect(str(DB_PATH), read_only=True)
rfm_summary = conn.execute("""
    SELECT
        segment_label,
        COUNT(customer_id) AS customer_count,
        ROUND(SUM(monetary), 2) AS total_spend,
        ROUND(AVG(monetary), 2) AS avg_spend,
        ROUND(AVG(frequency), 2) AS avg_frequency,
        ROUND(AVG(recency_days), 1) AS avg_recency_days
    FROM mart_rfm_segments
    GROUP BY segment_label
    ORDER BY total_spend DESC;
""").fetchdf()
display(rfm_summary)
conn.close()

## 4. Leakage-Free Repeat Buyer GMV Contribution
Comparing Lifetime Method A vs. Point-in-Time Method B (Order sequence >= 2 at transaction time).

In [ ]:
conn = duckdb.connect(str(DB_PATH), read_only=True)
with open(ROOT_DIR / "sql" / "07_repeat_buyer_gmv_contribution.sql") as f:
    query = f.read()
repeat_df = conn.execute(query).fetchdf()
display(repeat_df)
conn.close()